# Fire2Air Darwin — Assessment 3
## 04 Model 3 Count Prediction and Explainability — AWS Notebook
**Member:** Navodya Piumanthi

## Cell 1 — Imports and optional modelling packages
Loads the modelling, explainability and AWS dependencies. XGBoost and SHAP remain optional so the notebook can still run if either package is unavailable.

In [1]:
from pathlib import Path
import json
import joblib
import shutil
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_poisson_deviance,
)

import boto3

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    print("XGBoost unavailable:", exc)

try:
    import shap
    HAS_SHAP = True
except Exception as exc:
    HAS_SHAP = False
    print("SHAP unavailable:", exc)

print("XGBoost available:", HAS_XGBOOST)
print("SHAP available:", HAS_SHAP)


XGBoost available: True
SHAP available: True


## Cell 2 — Shared AWS S3 configuration and helper functions
Uses the same bucket/prefix structure as the other Fire2Air AWS scripts and adds reusable download, upload and listing helpers.

In [2]:
AWS_REGION = "ap-southeast-2"
BUCKET = "fire2air-darwin-dan5-646468992009-ap-southeast-2-an"

NTEPA_RAW = f"s3://{BUCKET}/raw/ntepa/"
FIRMS_RAW = f"s3://{BUCKET}/raw/firms/"
NTEPA_DAILY = f"s3://{BUCKET}/processed/ntepa_daily/"
FIRMS_DAILY = f"s3://{BUCKET}/processed/firms_daily/"
MODEL_READY = f"s3://{BUCKET}/processed/model_ready/"

MODEL1_DIR = f"s3://{BUCKET}/models/model1/"
MODEL2_DIR = f"s3://{BUCKET}/models/model2/"
MODEL3_DIR = f"s3://{BUCKET}/models/model3/"

MODEL1_OUTPUT = f"s3://{BUCKET}/outputs/model1/"
MODEL2_OUTPUT = f"s3://{BUCKET}/outputs/model2/"
MODEL3_OUTPUT = f"s3://{BUCKET}/outputs/model3/"

VIS_OUTPUT = f"s3://{BUCKET}/outputs/visualisation/"
DASHBOARD_DIR = f"s3://{BUCKET}/dashboard/"
DOCUMENTATION_DIR = f"s3://{BUCKET}/documentation/"
ATHENA_RESULTS = f"s3://{BUCKET}/athena-results/"

s3 = boto3.client("s3", region_name=AWS_REGION)

# Cross-platform temporary workspace: works on Windows, macOS and Linux.
LOCAL_ROOT = Path(tempfile.gettempdir()) / "fire2air_a3"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def _prefix(uri):
    return uri.replace(f"s3://{BUCKET}/", "", 1)

def download_prefix(uri, local_dir, flatten=False):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    prefix = _prefix(uri)
    paginator = s3.get_paginator("list_objects_v2")
    downloaded = []

    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue

            rel = Path(key[len(prefix):].lstrip("/"))
            dest = local_dir / (rel.name if flatten else rel)
            dest.parent.mkdir(parents=True, exist_ok=True)

            s3.download_file(BUCKET, key, str(dest))
            downloaded.append(dest)

    return downloaded

def upload_file(local_path, uri, object_name=None):
    local_path = Path(local_path)
    prefix = _prefix(uri).rstrip("/")
    key = f"{prefix}/{object_name or local_path.name}"
    s3.upload_file(str(local_path), BUCKET, key)
    print(f"Uploaded: s3://{BUCKET}/{key}")
    return key

def upload_tree(local_dir, uri):
    local_dir = Path(local_dir)
    if not local_dir.exists():
        return []

    prefix = _prefix(uri).rstrip("/")
    uploaded = []

    for p in local_dir.rglob("*"):
        if p.is_file():
            rel = p.relative_to(local_dir).as_posix()
            key = f"{prefix}/{rel}"
            s3.upload_file(str(p), BUCKET, key)
            uploaded.append(key)

    print(f"Uploaded {len(uploaded)} file(s): {local_dir} -> {uri}")
    return uploaded

def list_prefix(uri):
    prefix = _prefix(uri)
    paginator = s3.get_paginator("list_objects_v2")
    rows = []

    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("/"):
                continue
            rows.append({
                "Key": obj["Key"],
                "Size_Bytes": obj["Size"],
                "Last_Modified": obj["LastModified"],
            })

    return pd.DataFrame(rows)


## Cell 3 — AWS preflight, download shared model-ready data and initialise workspace
Checks S3 access, verifies the required shared checkpoint/manifest exist, downloads them, creates the local output folders, and loads the Model 3 target and feature manifest.

In [ ]:
RANDOM_STATE = 42
BOOTSTRAP_REPEATS = 300

project_folder = LOCAL_ROOT / "model3_workspace"
if project_folder.exists():
    shutil.rmtree(project_folder)
project_folder.mkdir(parents=True, exist_ok=True)

# Confirm the project bucket is accessible.
s3.head_bucket(Bucket=BUCKET)

model_ready_objects = list_prefix(MODEL_READY)
if model_ready_objects.empty:
    raise FileNotFoundError(f"No objects found under {MODEL_READY}")

required_files = {
    "Fire2Air_model_ready_checkpoint_A3.csv",
    "Fire2Air_feature_manifest_A3.json",
}
available_names = {Path(key).name for key in model_ready_objects["Key"]}
missing_required = required_files.difference(available_names)

if missing_required:
    raise FileNotFoundError(
        f"Required Model 3 input file(s) missing from {MODEL_READY}: "
        f"{sorted(missing_required)}"
    )

print("Model-ready S3 objects:")
display(model_ready_objects)

downloaded_files = download_prefix(
    MODEL_READY,
    project_folder,
    flatten=True,
)
print(f"Downloaded {len(downloaded_files)} file(s) to:", project_folder)

output_root = project_folder / "outputs_prt661_a3"
table_dir = output_root / "tables"
figure_dir = output_root / "figures"
model_dir = output_root / "models"

for folder in [table_dir, figure_dir, model_dir]:
    folder.mkdir(parents=True, exist_ok=True)

data_path = project_folder / "Fire2Air_model_ready_checkpoint_A3.csv"
manifest_path = project_folder / "Fire2Air_feature_manifest_A3.json"

integrated_daily = pd.read_csv(
    data_path,
    parse_dates=["date", "target_date"],
)

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

model_features = manifest["features"]
feature_groups = manifest["feature_groups"]
duration_target = manifest["targets"]["count"]

print("AWS staging folder:", project_folder)
print("Dataset shape:", integrated_daily.shape)
print("Predictor count:", len(model_features))
print("Count target:", duration_target)
